# 01 — Web Scraping
**FIFA 2026 World Cup Predictor**

This notebook scrapes all historical World Cup match results (1930–2022) from Wikipedia,  
plus the full 2026 fixture and group tables.

## Install dependencies

In [1]:
!pip install beautifulsoup4 requests lxml pandas -q

## Imports

In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time

HEADERS = {"User-Agent": "FIFA2026Predictor/1.0 (educational project)"}

YEARS = [
    1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974,
    1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014,
    2018, 2022,
]
print(f"Will scrape {len(YEARS)} World Cups: {YEARS[0]}–{YEARS[-1]}")

Will scrape 22 World Cups: 1930–2022


## `get_matches()` — scrape one World Cup year

In [3]:
def get_matches(year: int) -> pd.DataFrame:
    """Scrape all match results for a given World Cup year from Wikipedia."""
    url = f"https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup"
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "lxml")

    home, score, away = [], [], []
    for match in soup.find_all("div", class_="footballbox"):
        home.append(match.find("th", class_="fhome").get_text(strip=True))
        score.append(match.find("th", class_="fscore").get_text(strip=True))
        away.append(match.find("th", class_="faway").get_text(strip=True))

    df = pd.DataFrame({"Home": home, "Score": score, "Away": away})
    df["year"] = year
    return df

# Quick test
sample = get_matches(2022)
print(f"2022: {len(sample)} matches")
sample.head()

2022: 64 matches


,Home,Score,Away,year
0,Qatar,0–2,Ecuador,2022
1,Senegal,0–2,Netherlands,2022
2,Qatar,1–3,Senegal,2022
3,Netherlands,1–1,Ecuador,2022
4,Ecuador,1–2,Senegal,2022


## Scrape all years → `data/raw_historical_data.csv`

In [4]:
frames = []
for year in YEARS:
    try:
        df = get_matches(year)
        frames.append(df)
        print(f"  ✓ {year}: {len(df)} matches")
        time.sleep(0.5)   # be polite to Wikipedia's servers
    except Exception as e:
        print(f"  ✗ {year}: {e}")

df_historical = pd.concat(frames, ignore_index=True)
df_historical.to_csv("data/raw_historical_data.csv", index=False)
print(f"\nTotal: {len(df_historical)} rows saved → data/raw_historical_data.csv")
df_historical.tail()

  ✓ 1930: 18 matches
  ✓ 1934: 17 matches
  ✓ 1938: 19 matches
  ✓ 1950: 22 matches
  ✓ 1954: 26 matches
  ✓ 1958: 35 matches
  ✓ 1962: 32 matches
  ✓ 1966: 32 matches
  ✓ 1970: 32 matches
  ✓ 1974: 38 matches
  ✓ 1978: 38 matches
  ✓ 1982: 52 matches
  ✓ 1986: 52 matches
  ✓ 1990: 16 matches
  ✓ 1994: 52 matches
  ✓ 1998: 64 matches
  ✓ 2002: 64 matches
  ✓ 2006: 64 matches
  ✓ 2010: 64 matches
  ✓ 2014: 64 matches
  ✓ 2018: 64 matches
  ✓ 2022: 64 matches

Total: 929 rows saved → data/raw_historical_data.csv


,Home,Score,Away,year
924,England,1–2,France,2022
925,Argentina,3–0,Croatia,2022
926,France,2–0,Morocco,2022
927,Croatia,2–1,Morocco,2022
928,Argentina,3–3(a.e.t.),France,2022


## Scrape 2026 fixture & group tables

In [5]:
# 2026 fixture (uses Wikipedia archived snapshot for stability)
ARCHIVE_URL = "https://web.archive.org/web/20260115040107/https://en.wikipedia.org/wiki/2026_FIFA_World_Cup"

import pickle
from string import ascii_uppercase as alphabet

all_tables = pd.read_html(ARCHIVE_URL)
print(f"Found {len(all_tables)} tables on the 2026 page")

Found 171 tables on the 2026 page


In [6]:
# Extract 12 group tables (Groups A–L)
dict_table = {}
for letter, i in zip(alphabet, range(26, 107, 7)):
    df = all_tables[i].copy()
    df.rename(columns={df.columns[1]: "Team"}, inplace=True)
    if "Qualification" in df.columns:
        df.pop("Qualification")
    dict_table[f"Group {letter}"] = df

print("Groups extracted:")
for g, tbl in dict_table.items():
    print(f"  {g}: {list(tbl['Team'].values)}")

with open("data/dict_table_2026", "wb") as f:
    pickle.dump(dict_table, f)
print("\nSaved → data/dict_table_2026")

Groups extracted:
  Group A: ['Mexico (H)', 'South Africa', 'South Korea', 'UEFA Path D winner[a]']
  Group B: ['Canada (H)', 'UEFA Path A winner[a]', 'Qatar', 'Switzerland']
  Group C: ['Brazil', 'Morocco', 'Haiti', 'Scotland']
  Group D: ['United States (H)', 'Paraguay', 'Australia', 'UEFA Path C winner[a]']
  Group E: ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador']
  Group F: ['Netherlands', 'Japan', 'UEFA Path B winner[a]', 'Tunisia']
  Group G: ['Belgium', 'Egypt', 'Iran', 'New Zealand']
  Group H: ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay']
  Group I: ['France', 'Senegal', 'IC Path 2 winner[a]', 'Norway']
  Group J: ['Argentina', 'Algeria', 'Austria', 'Jordan']
  Group K: ['Portugal', 'IC Path 1 winner[a]', 'Uzbekistan', 'Colombia']
  Group L: ['England', 'Croatia', 'Ghana', 'Panama']

Saved → data/dict_table_2026


In [7]:
# Scrape 2026 fixture
df_fixture_2026 = get_matches(2026)
df_fixture_2026.to_csv("data/raw_fixture_2026.csv", index=False)
print(f"{len(df_fixture_2026)} fixture rows saved → data/raw_fixture_2026.csv")
df_fixture_2026.head(10)

104 fixture rows saved → data/raw_fixture_2026.csv


,Home,Score,Away,year
0,Mexico,Match 1,South Africa,2026
1,South Korea,Match 2,UEFA Path Dwinner,2026
2,UEFA Path Dwinner,Match 25,South Africa,2026
3,Mexico,Match 28,South Korea,2026
4,UEFA Path Dwinner,Match 53,Mexico,2026
5,South Africa,Match 54,South Korea,2026
6,Canada,Match 3,UEFA Path Awinner,2026
7,Qatar,Match 8,Switzerland,2026
8,Switzerland,Match 26,UEFA Path Awinner,2026
9,Canada,Match 27,Qatar,2026


## Summary
| File | Description |
|---|---|
| `data/raw_historical_data.csv` | Raw scraped results, 1930–2022 |
| `data/raw_fixture_2026.csv` | Raw 2026 fixture |
| `data/dict_table_2026` | Group table structures (pickle) |
